
# AgroUTMACH PIX
## NB02 — Entrenamiento principal YOLO11s sobre PIO

**Objetivo:** entrenar el detector principal del proyecto de feria con el dataset PIO ya auditado.

### Configuración propuesta
- Modelo: **YOLO11s preentrenado**
- Resolución: **960 × 960**
- Épocas máximas: **60**
- *Early stopping*: **15 épocas**
- Optimizador: **AdamW**
- Semilla: **2026**
- GPU recomendada: **NVIDIA L4**
- Dataset: 1 clase — `Pollo`

> Este notebook copia temporalmente el dataset desde Drive al almacenamiento local de Colab para acelerar el entrenamiento, pero guarda los resultados y pesos importantes en Drive.



## 0. Antes de ejecutar

En Google Colab seleccione:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**

Si está disponible, use **L4**.

Ejecute las celdas **en orden**.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

# Verificar GPU
!nvidia-smi


In [ ]:

!pip -q install -U ultralytics


In [ ]:

from pathlib import Path
import os, shutil, json, time, yaml
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM total: {props.total_memory/1024**3:.1f} GB")
else:
    raise RuntimeError("No hay GPU activa. Cambie el entorno de ejecución a GPU antes de continuar.")


## 1. Definir rutas del proyecto

In [ ]:

ROOT = Path("/content/drive/MyDrive/AgroUTMACH_PIX")
DATASET_DRIVE = ROOT / "01_DATASET_PIO" / "PIO_TRAIN_VAL" / "data"
MODELS_DIR = ROOT / "03_MODELOS"
RESULTS_DIR = ROOT / "05_RESULTADOS"
RUNS_DIR = RESULTS_DIR / "NB02_YOLO11S_960"

LOCAL_DATA = Path("/content/PIO_LOCAL")
LOCAL_YAML = LOCAL_DATA / "dataset_colab.yaml"

for p in [MODELS_DIR, RESULTS_DIR, RUNS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert (DATASET_DRIVE / "images" / "train").exists(), "No se encontró images/train"
assert (DATASET_DRIVE / "images" / "val").exists(), "No se encontró images/val"
assert (DATASET_DRIVE / "labels" / "train").exists(), "No se encontró labels/train"
assert (DATASET_DRIVE / "labels" / "val").exists(), "No se encontró labels/val"

print("✓ Rutas verificadas")
print("Dataset en Drive:", DATASET_DRIVE)
print("Resultados:", RUNS_DIR)



## 2. Verificación rápida del dataset

NB01 encontró:
- **1.035** imágenes de entrenamiento
- **452** imágenes de validación
- **1.487** imágenes en total
- una sola clase: `Pollo`

También detectó `classes.txt` dentro de `labels/train`, que **no es una anotación YOLO**. Se excluirá de la copia local.


In [ ]:

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

n_train = count_images(DATASET_DRIVE / "images" / "train")
n_val = count_images(DATASET_DRIVE / "images" / "val")

print(f"Train: {n_train:,}")
print(f"Val:   {n_val:,}")
print(f"Total: {n_train+n_val:,}")

assert n_train == 1035, f"Se esperaban 1035 imágenes train y se encontraron {n_train}"
assert n_val == 452, f"Se esperaban 452 imágenes val y se encontraron {n_val}"
print("✓ Conteos correctos.")



## 3. Copiar dataset al disco local de Colab

Esto evita entrenar directamente leyendo 1.487 imágenes desde Google Drive en cada época.


In [ ]:

if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

print("Copiando dataset desde Drive a /content ...")
t0 = time.time()

shutil.copytree(DATASET_DRIVE / "images", LOCAL_DATA / "images")
shutil.copytree(DATASET_DRIVE / "labels", LOCAL_DATA / "labels")

# El archivo classes.txt no corresponde a una imagen y no forma parte del formato de anotación YOLO.
extra_classes = LOCAL_DATA / "labels" / "train" / "classes.txt"
if extra_classes.exists():
    extra_classes.unlink()
    print("✓ classes.txt eliminado de la copia local (original en Drive intacto).")

elapsed = time.time() - t0
print(f"✓ Copia local completada en {elapsed/60:.1f} min")
print(f"Train local: {count_images(LOCAL_DATA/'images'/'train'):,}")
print(f"Val local:   {count_images(LOCAL_DATA/'images'/'val'):,}")


## 4. Crear YAML correcto para Google Colab

In [ ]:

dataset_cfg = {
    "path": str(LOCAL_DATA),
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["Pollo"],
}

with open(LOCAL_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(dataset_cfg, f, sort_keys=False, allow_unicode=True)

print(LOCAL_YAML.read_text())



## 5. Configuración del entrenamiento principal

**Por qué YOLO11s a 960 px:** el conjunto contiene escenas muy densas y muchos animales pequeños. Una resolución mayor ayuda a conservar más detalle espacial. `YOLO11s` ofrece un buen equilibrio entre calidad y velocidad para la demostración de feria.

Si Colab reporta falta de memoria, cambie únicamente:

```python
BATCH = 4
```


In [ ]:

MODEL_NAME = "yolo11s.pt"
IMGSZ = 960
EPOCHS = 60
BATCH = 8
PATIENCE = 15
SEED = 2026

print("CONFIGURACIÓN")
print("Modelo:", MODEL_NAME)
print("Imagen:", IMGSZ)
print("Épocas máximas:", EPOCHS)
print("Batch:", BATCH)
print("Patience:", PATIENCE)
print("Seed:", SEED)


## 6. Entrenar YOLO11s

In [ ]:

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(LOCAL_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.002,
    momentum=0.9,
    weight_decay=0.0005,
    cos_lr=True,
    patience=PATIENCE,
    pretrained=True,
    amp=True,
    seed=SEED,
    deterministic=True,
    close_mosaic=10,
    plots=True,
    save=True,
    save_period=5,
    project=str(RUNS_DIR),
    name="train",
    exist_ok=True,
    verbose=True,
)

print("\n✓ ENTRENAMIENTO FINALIZADO")


## 7. Localizar el mejor modelo

In [ ]:

BEST = RUNS_DIR / "train" / "weights" / "best.pt"
LAST = RUNS_DIR / "train" / "weights" / "last.pt"

assert BEST.exists(), f"No se encontró {BEST}"

FINAL_MODEL = MODELS_DIR / "AgroUTMACH_PIX_YOLO11s_PIO_best.pt"
shutil.copy2(BEST, FINAL_MODEL)

print("✓ best.pt:", BEST)
print("✓ Copia final:", FINAL_MODEL)
if LAST.exists():
    print("✓ last.pt:", LAST)


## 8. Validación final del mejor modelo

In [ ]:

best_model = YOLO(str(FINAL_MODEL))

metrics = best_model.val(
    data=str(LOCAL_YAML),
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name="val_best",
    exist_ok=True
)

summary = {
    "model": MODEL_NAME,
    "imgsz": IMGSZ,
    "epochs_max": EPOCHS,
    "batch": BATCH,
    "seed": SEED,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
}

print("\nMÉTRICAS DEL MODELO")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"{k:12s}: {v:.4f}")
    else:
        print(f"{k:12s}: {v}")

with open(RESULTS_DIR / "NB02_metricas_YOLO11s_960.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n✓ Métricas guardadas en Drive.")


## 9. Predicciones visuales sobre imágenes de validación

In [ ]:

VAL_DIR = LOCAL_DATA / "images" / "val"

pred = best_model.predict(
    source=str(VAL_DIR),
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.6,
    device=0,
    save=True,
    max_det=1000,
    project=str(RUNS_DIR),
    name="pred_val",
    exist_ok=True,
    stream=True,
)

# Procesar solo 12 imágenes para la galería de revisión.
for i, r in enumerate(pred):
    if i >= 11:
        break

print("✓ Se generó una muestra visual de predicciones.")
print("Ruta:", RUNS_DIR / "pred_val")



## 10. Resultado de esta etapa

Al finalizar correctamente, debe existir:

```text
AgroUTMACH_PIX/
├── 03_MODELOS/
│   └── AgroUTMACH_PIX_YOLO11s_PIO_best.pt
│
└── 05_RESULTADOS/
    ├── NB02_YOLO11S_960/
    │   ├── train/
    │   ├── val_best/
    │   └── pred_val/
    └── NB02_metricas_YOLO11s_960.json
```

**No continúe todavía con el video.**  
Primero revisaremos las métricas y las predicciones. Después construiremos **NB03 — detección y conteo sobre `0002.mp4`**.


In [ ]:

print("=" * 70)
print("NB02 FINALIZADO CORRECTAMENTE")
print("Modelo final:", FINAL_MODEL)
print(f"mAP@50:    {summary['mAP50']:.4f}")
print(f"mAP@50-95: {summary['mAP50_95']:.4f}")
print(f"Precision: {summary['precision']:.4f}")
print(f"Recall:    {summary['recall']:.4f}")
print("=" * 70)
